In [ ]:
# Data Loading Using SnowSQL
# This notebook demonstrates loading a sample CSV dataset into Snowflake
# using the same PUT + COPY INTO pattern that SnowSQL uses.

import csv
import os

# Create a sample CSV dataset
sample_data = [
    ['EMP_ID', 'EMP_NAME', 'DEPARTMENT', 'SALARY', 'HIRE_DATE'],
    [101, 'John Smith', 'Engineering', 85000, '2022-01-15'],
    [102, 'Sarah Connor', 'Marketing', 72000, '2021-06-20'],
    [103, 'Mike Johnson', 'Engineering', 92000, '2020-03-10'],
    [104, 'Emily Davis', 'HR', 68000, '2023-02-01'],
    [105, 'Robert Brown', 'Marketing', 75000, '2022-08-12'],
    [106, 'Lisa Wilson', 'Engineering', 98000, '2019-11-05'],
    [107, 'David Lee', 'HR', 71000, '2021-09-30'],
    [108, 'Anna Taylor', 'Finance', 88000, '2020-07-22'],
    [109, 'James White', 'Finance', 95000, '2018-04-18'],
    [110, 'Maria Garcia', 'Marketing', 69000, '2023-05-14'],
]

# Write to CSV file
csv_path = '/tmp/employees.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(sample_data)

print(f'Sample CSV created at {csv_path}')
print(f'Records: {len(sample_data) - 1} rows (excluding header)')

In [ ]:
%%sql -r setup_db
-- Step 1: Create the target database, schema, and table
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS SNOWFLAKE_ASSIGNMENT;
USE DATABASE SNOWFLAKE_ASSIGNMENT;
CREATE SCHEMA IF NOT EXISTS ASSIGNMENT_SCHEMA;
USE SCHEMA ASSIGNMENT_SCHEMA;

In [ ]:
%%sql -r create_emp_table
-- Step 2: Create the target table for loading
CREATE OR REPLACE TABLE EMPLOYEES (
    EMP_ID INT,
    EMP_NAME VARCHAR(100),
    DEPARTMENT VARCHAR(50),
    SALARY NUMBER(10,2),
    HIRE_DATE DATE
);

In [ ]:
%%sql -r create_ff
-- Step 3: Create a file format for CSV loading
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
    TYPE = 'CSV'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1;

In [ ]:
%%sql -r create_stg
-- Step 4: Create an internal stage (equivalent to SnowSQL staging area)
CREATE OR REPLACE STAGE EMP_STAGE
    FILE_FORMAT = CSV_FORMAT;

In [ ]:
# Step 5: PUT file to stage (equivalent to SnowSQL: PUT file://employees.csv @EMP_STAGE)
from snowflake.snowpark.context import get_active_session

session = get_active_session()
put_result = session.sql("PUT file:///tmp/employees.csv @EMP_STAGE AUTO_COMPRESS=TRUE").collect()

for row in put_result:
    print(f"File: {row['source']} | Target: {row['target']} | Status: {row['status']}")

In [ ]:
%%sql -r list_stage
-- Verify file is in the stage
LIST @EMP_STAGE;

In [ ]:
%%sql -r copy_result
-- Step 6: COPY INTO table from stage (the actual data load)
COPY INTO EMPLOYEES
    FROM @EMP_STAGE
    FILE_FORMAT = CSV_FORMAT
    ON_ERROR = 'CONTINUE';

## Verify Data Load

In [ ]:
%%sql -r row_count
-- Verify: Row count
SELECT COUNT(*) AS TOTAL_ROWS FROM EMPLOYEES;

In [ ]:
%%sql -r all_employees
-- Verify: View all loaded records
SELECT * FROM EMPLOYEES ORDER BY EMP_ID;

In [ ]:
%%sql -r dept_summary
-- Verify: Summary by department
SELECT 
    DEPARTMENT,
    COUNT(*) AS EMP_COUNT,
    AVG(SALARY)::NUMBER(10,2) AS AVG_SALARY
FROM EMPLOYEES
GROUP BY DEPARTMENT
ORDER BY EMP_COUNT DESC;